# 01 — MuJoCo Playground で四脚（Go1）の歩行を学習する

テスト機 `Go1JoystickFlatTerrain` を Brax PPO で学習し、歩く動画を出すまで。自作四脚に進む前の練習台。

**準備**: ランタイム → ランタイムのタイプを変更 → **GPU（T4 以上）**。上から順に実行する。

ロジックは `quadleg_rl/train.py` にあり、このノートブックは呼び出すだけ。コードの編集は Zed で行い、GitHub に push → セル 2 で pull。

## 実行実績（T4、2026-09-06）

| 段階 | ステップ | 時間 | 最終 reward | 結果 |
| --- | --- | --- | --- | --- |
| 初回 | 2,949 万 | 15 分 | 17.1 | **指令を無視してその場に立つ**（局所解） |
| 継続 | +7,373 万（累計 約 1 億） | 23 分 | 26.9 | 前進 1.0 m/s 指令に 0.92 m/s で追従、旋回も可 |

reward は 2,000 万付近から 18 前後で長く停滞し、累計 6,300 万あたりで 19.9 → 23.0 と階段状に跳ねて歩き出す。
**途中で頭打ちに見えても学習量不足を疑うこと**（公式推奨は 2 億ステップ）。

In [ ]:
#@title 1. GPU 確認
!nvidia-smi -L

In [ ]:
#@title 2. リポジトリ取得
REPO_URL = "https://github.com/yosihitoyasudasub/quadleg-rl.git"  #@param {type:"string"}
REPO_DIR = "/content/quadleg-rl"
import os, sys, importlib
if os.path.isdir(REPO_DIR):
    !cd $REPO_DIR && git pull
else:
    !git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()   # clone 前にパスを通していると「中身なし」がキャッシュされる
import quadleg_rl
print("OK:", quadleg_rl.__file__)

In [ ]:
#@title 3. インストール（初回 2〜3 分）
# flax は Colab のプリインストール版が古く、JAX が移動した jax.core.get_opaque_trace_state を
# 直接呼ぶため明示的に更新する。
# brax は PyPI 最新の 0.14.2 が JAX 0.10 で削除された jax.device_put_replicated を使うが、
# quadleg_rl/train.py が互換シムを持っているので PyPI 版のままでよい
#（GitHub main では修正済み。新しい brax がリリースされたらシムは不要になる）。
%pip install -q -U "jax[cuda12]" playground mediapy "flax>=0.12"
import jax, flax, brax, mujoco
print("jax", jax.__version__, "| flax", flax.__version__, "| brax", brax.__version__, "| mujoco", mujoco.__version__)
print("backend:", jax.default_backend(), jax.devices())
# 「RESTART SESSION」を促されたら再起動し、セル 2 から実行し直す

In [ ]:
#@title 4. Google Drive をマウント（チェックポイント・動画の保存先）
from google.colab import drive
drive.mount("/content/drive")
LOGDIR = "/content/drive/MyDrive/quadleg-rl/logs"
import os; os.makedirs(LOGDIR, exist_ok=True); print(LOGDIR)

In [ ]:
#@title 5. 学習（T4: 3,000 万ステップで約 15 分。JIT に最初の 2.5 分）
# num_evals で割った 1 エポック分に切り上げられるので、実際のステップ数は指定より多くなる。
# 歩行までは累計 1 億ステップ程度が必要。まず 3,000 万で流れを確認し、セル 6 で継続する。
ENV_NAME = "Go1JoystickFlatTerrain"  #@param ["Go1JoystickFlatTerrain", "Go1JoystickRoughTerrain", "Go1Getup", "SpotFlatTerrainJoystick", "BarkourJoystick"]
NUM_TIMESTEPS = 30_000_000  #@param {type:"integer"}
NUM_EVALS = 10  #@param {type:"integer"}
DOMAIN_RANDOMIZATION = True  #@param {type:"boolean"}

import os, sys, importlib
REPO_DIR = globals().get("REPO_DIR", "/content/quadleg-rl")
if not os.path.isdir(os.path.join(REPO_DIR, "quadleg_rl")):
    raise RuntimeError(f"{REPO_DIR}/quadleg_rl が無い。セル 2 を実行してください。")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()
if "LOGDIR" not in globals():
    raise RuntimeError("LOGDIR 未定義。セル 4（Drive マウント）を実行してください。")

from quadleg_rl import train
result = train.train(ENV_NAME, num_timesteps=NUM_TIMESTEPS, num_evals=NUM_EVALS,
                     logdir=LOGDIR, domain_randomization=DOMAIN_RANDOMIZATION)
train.plot_history(result)

In [ ]:
#@title 6. 学習の継続（前の run のチェックポイントから再開）
# 空欄なら LOGDIR 内の最新の run を使う。切断されたときの復旧にも同じ手順が使える。
# 開始直後の [0] reward が前回の最終値と一致すれば復元成功（0.002 付近ならゼロから学習し直している）。
RESUME_RUN = ""  #@param {type:"string"}
MORE_TIMESTEPS = 60_000_000  #@param {type:"integer"}

import os, sys, importlib
REPO_DIR = globals().get("REPO_DIR", "/content/quadleg-rl")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()
from quadleg_rl import train                      # 再起動後にこのセルから始めても動くようにする
ENV_NAME = globals().get("ENV_NAME", "Go1JoystickFlatTerrain")
DOMAIN_RANDOMIZATION = globals().get("DOMAIN_RANDOMIZATION", True)

from pathlib import Path
runs = sorted([p for p in Path(LOGDIR).iterdir() if (p / "checkpoints").is_dir()])
run_dir = Path(RESUME_RUN) if RESUME_RUN else runs[-1]
print("resume from:", run_dir)

result = train.train(ENV_NAME, num_timesteps=MORE_TIMESTEPS, num_evals=10, logdir=LOGDIR,
                     domain_randomization=DOMAIN_RANDOMIZATION,
                     restore_checkpoint_path=str(run_dir / "checkpoints"))
train.plot_history(result)

In [ ]:
#@title 7. 指令追従の診断（描画なし・約 1 分）
# 4 種類の指令でロールアウトし、指令どおり動いているかを数値で確認する。
#   vx_body_mps  … 機体前方の速度。前進指令の値に近ければ追従できている
#   yaw_rate     … 旋回角速度（rad/s）
#   tracking_lin_vel / tracking_ang_vel … 報酬メトリクス（それぞれ最大 1.0 / 0.5）
# 全指令で速度がほぼ 0 なら「立っているだけ」の局所解。セル 6 で学習量を足す。
rows = train.probe(result)

In [ ]:
#@title 8. 歩行動画（前進 1.0 m/s）
# 描画は 1 フレームあたり 1 秒近くかかる。EPISODE_LENGTH=500（10 秒）で 4 分程度。
EPISODE_LENGTH = 500  #@param {type:"integer"}
VX = 1.0  #@param {type:"number"}

import mediapy as media
path = train.render_video(result, result.logdir / f"walk_vx{VX}.mp4",
                          command=(VX, 0.0, 0.0), episode_length=EPISODE_LENGTH)
media.show_video(media.read_video(path), fps=25)

In [ ]:
#@title 9. 旋回動画（ヨー 1.0 rad/s）
YAW = 1.0  #@param {type:"number"}
path = train.render_video(result, result.logdir / f"turn_yaw{YAW}.mp4",
                          command=(0.0, 0.0, YAW), episode_length=EPISODE_LENGTH)
media.show_video(media.read_video(path), fps=25)

## トラブルシュート（このノートブックで実際に踏んだもの）

| 症状 | 原因と対処 |
| --- | --- |
| `ModuleNotFoundError: quadleg_rl` | clone より先に `sys.path` へ入れると importlib が「中身なし」をキャッシュする。セル 2 の `importlib.invalidate_caches()` で解消 |
| `AttributeError: type object 'int' has no attribute 'WARP'` | Playground の Go1 は既定 config が `impl="warp"` だが `mujoco-warp` が入っていない。`train.train(impl="jax")`（既定）で回避 |
| `jax.core.get_opaque_trace_state ... removed` | Colab の古い flax。セル 3 の `flax>=0.12` で更新 |
| `jax.device_put_replicated is deprecated` | brax 0.14.2 が JAX 0.10 で削除された API を使う。`train.py` の互換シムが肩代わりする |
| 動画で機体が端に寄って動かない | `camera="track"` が未指定だった（現在は既定）。加えて学習不足で本当に止まっていた |
| ランタイム再起動後に動かない | pip はディスク、import はメモリ。**install → 再起動 → import** の順を守る。再起動すると `result` は失われるのでセル 6 で復元する |

## 運用メモ

- Colab 無料枠は 90 分無操作で切断、最大 12 時間。長時間学習では PC をスリープさせない（Win+L のロックだけなら問題ない）
- 切断されてもチェックポイントは Drive に残るので、セル 6 で再開できる
- T4 で約 68,000 ステップ/秒。1 億ステップ ≒ 25 分（JIT を除く）

## 次のステップ

- `quadleg_rl/envs/go1_2dof.py`: Go1 の外転関節を固定した 8 アクチュエータ環境（実機の 4 脚 × 2 自由度と同条件）。平面脚で歩行と差動ステア旋回ができるかを検証する
- 自作四脚の MJCF 生成と学習（`03_custom_quadruped.ipynb` 予定）。**四脚は現在保留中**で、再開手順は `docs/quadruped.md` にある
- 詳細は `docs/HANDOFF.md`